### Следующая ячейка - блок импорта общих библиотек и формирования переменных
#### должна выполняться первой

In [1]:
import os # Отключаем попытку tqdm найти визуальный виджет IProgress 
#from tqdm.auto import tqdm
os.environ['TQDM_DISABLE'] = '1'

In [2]:
# ИМПОРТ БИБЛИОТЕК
import os
from collections import Counter
from pathlib import Path
import httpx
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
# Загружаем переменные из модуля  ai_engineer_preproject\path.py
from ai_engineer_preproject.path import (
    PROJECT_ROOT, 
    DOCS_DIR, 
    LOCAL_EMBED_MODEL, 
    CHUNK_EMBEDINGS,
    OLLAMA_URL,
    INDEXES,
    LLM_MODEL, 
)

# Получаем путь к папке с документами (данные для LLM)
print(f"Корнева папка проекта:\n{PROJECT_ROOT}")
print(f"Папка с документами датасета:\n{DOCS_DIR}")
print(f"Путь к папке локально сохраненной модели эмбеддингов:\n{LOCAL_EMBED_MODEL}")
print(f"Путь к файлу с локально сохраненными эмбедингами чанков:\n{CHUNK_EMBEDINGS}")
print(f"URL сервера ollama:\n{OLLAMA_URL}")

Корнева папка проекта:
/home/wirekit69/ai_engineer_preproject
Папка с документами датасета:
/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs
Путь к папке локально сохраненной модели эмбеддингов:
/home/wirekit69/ai_engineer_preproject_DATA/local_embed_model
Путь к файлу с локально сохраненными эмбедингами чанков:
/home/wirekit69/ai_engineer_preproject_DATA/chunk_embeddings/chunk_embeddings.npy
URL сервера ollama:
http://localhost:11434


In [3]:
# Готовая функция чанкинга 
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [4]:
# Проверяем работоспособность сервера ollama и моделей
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

Найдено моделей: 5
 - qwen2.5-coder:1.5b
 - llama3.1:8b
 - qwen2.5-coder:7b
 - lfm3:1.7b
 - qwen2.5:1.5b


In [5]:
for path in sorted(DOCS_DIR.rglob("*"))[:5]:
    print(f"{str(path)} -> {path.name}")


/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/README.md -> README.md
/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api -> api
/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api/anthropic-compatibility.mdx -> anthropic-compatibility.mdx
/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api/authentication.mdx -> authentication.mdx
/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api/balance.mdx -> balance.mdx


In [6]:
# ЗАГРУЖАЕМ ДОКУМЕНТЫ
documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

Загружено документов: 70


In [7]:
# НАРЕЗАЕМ НА ЧАНКИ 
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 609
Средняя длина чанка: 540 символов


In [8]:
# Посмотрим на несколько примеров чанков
for chunk in chunks[200:202]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из /home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/capabilities/structured-outputs.mdx (569 символов) ---
## Example: Extract structured data

Define the objects you want returned and let the model populate the fields:

```python
from ollama import chat
from pydantic import BaseModel

class Pet(BaseModel) ...

--- из /home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/capabilities/structured-outputs.mdx (582 символов) ---
## Example: Vision with structured outputs

Vision models accept the same `format` parameter, enabling deterministic descriptions of images:

```python
from ollama import chat
from pydantic import Bas ...



In [9]:
# СВОДКА ПО ЧАНКАМ
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Всего чанков: 609
Файлов с чанками: 70
Топ-3 файла по числу чанков:
  /home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api.md: 74 чанков
  /home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/faq.mdx: 38 чанков
  /home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/capabilities/tool-calling.mdx: 33 чанков


In [10]:
file_path = DOCS_DIR / "api.md"
sample_text = file_path.read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    avg_length = sum(len(c) for c in chunks_sample) // len(chunks_sample) if chunks_sample else 0
    print(f"max_chars = {max_chars}\n-> чанков: {len(chunks_sample)}, средняя длина: {avg_length}")


max_chars = 200
-> чанков: 161, средняя длина: 330
max_chars = 800
-> чанков: 74, средняя длина: 736
max_chars = 3000
-> чанков: 30, средняя длина: 1818


In [ ]:
# Загрузка модели с сайта HF
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)



In [11]:
# ===!!!=== ТОЛЬКО ДЛЯ ЗАГРУЗКИ ЛОКАЛЬНО СОХРАНЕННОЙ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!===
embed_model = SentenceTransformer(LOCAL_EMBED_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# ===!!!=== ТОЛЬКО ДЛЯ ЛОКАЛЬНОГО СОХРАНЕНИЯ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!=== 
# folder for save model = {}
# Файловый размер модели около 450 мегабайт
print(f"Сохраняем модель эмбедер в папку:\n{LOCAL_EMBED_MODEL}")
embed_model.save(LOCAL_EMBED_MODEL)

In [12]:
# ФОРМИРОВАНИЕ ЕМБЕДДИНГОВ
# # Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (609, 384)


In [ ]:
# ЛОКАЛЬНОЕ СОХРАНЕНИЕ ГОТОВЫХ ЕМБЕДДИНГОВ
np.save(CHUNK_EMBEDINGS, chunk_embeddings)

In [ ]:
# ЗАГРУЗКА ЕМЕБЕДДИНГОВ ИЗ ЛОКАЛЬНО СОХРАНЕННОГО ФАЙЛА
chunk_embeddings = np.load(CHUNK_EMBEDINGS)

In [13]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quic...
Его эмбеддинг (первые 10 чисел): [-0.06130737 -0.0551094  -0.03115822 -0.01005447  0.03433182  0.04836938
 -0.0826564   0.02022986 -0.00097113  0.04949645]
Длина вектора (норма): 1.0000


In [14]:
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"В индексе {index.ntotal} векторов размерности {dim}") 

В индексе 609 векторов размерности 384


In [15]:
print(INDEXES)

/home/wirekit69/ai_engineer_preproject_DATA/local_indexes/MY_index.faiss


In [ ]:
# ===!!!=== ЛОКАЛЬНОЕ СОХРАНЕНИЕ ИНДЕКСОВ FAISS В ФАЙЛ ===!!!===
# Запускать ячейку только для локального сохранение модели индексов
faiss.write_index(index, INDEXES.as_posix())

In [16]:
# ===!!!=== ЗАГРУЗКА ИНДЕКСОВ FAISS ИЗ ФАЙЛА ===!!!===
# Запускать ячейку только для локальной загрузки модели индексов
index = faiss.read_index(INDEXES.as_posix())

In [17]:
# Функция для поиска ближайших чанков по смыслу
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [18]:
question = "Как сменить папку, в которой Ollama хранит модели?"


In [19]:
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, в которой Ollama хранит модели?

#1  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/windows.mdx]  score=0.765
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#2  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/macos.mdx]  score=0.629
## Filesystem Requirements

The preferred method of installation is to mount the `ollama.dmg` and drag-and-drop the Ollama application to the system-wide `Applications` folder.  Upon startup, the Olla ...

#3  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/faq.mdx]  score=0.620
Alternatively, you can change the amount of time all models are loaded into memory by setting the `OLLAMA_KEEP_ALIVE` environment variable when starting the Ollama server. The `OLLAMA_KEEP_ALIVE` vari ...



In [20]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    results = vector_search(q, top_k=3) 
    for r in results:
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:150].strip()}...\n") 



=== What HTTP method does /api/generate use? ===
  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/api/cloud-usage.mdx]  score=0.467
  ---
title: Cloud usage
sidebarTitle: /api/usage
openapi: /openapi.yaml GET /api/usage
---

<RequestExample>

```shell Request
curl 'https://ollama.com...

  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/faq.mdx]  score=0.461
  ```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollama-with-ca
```...

  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/quickstart.mdx]  score=0.454
  These examples use the `gemma4:31b` [cloud model](https://ollama.com/search?c=cloud).

<Tabs>
  <Tab title="Ollama API">

```shell
curl https://ollama...

=== Как сменить папку, где хранятся модели? ===
  [/home/wirekit69/ai_engineer_preproject_DATA/ollama_docs/faq.mdx]  score=0.586
  ## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- W

In [22]:
LLM_MODEL

'qwen2.5:1.5b'

In [23]:
import httpx

response = httpx.post(
    "http://localhost:11434/api/generate",
    json={
        "model": LLM_MODEL,
        "prompt": "Привет! Назови одно интересное свойство Python в одном предложении.",
        "stream": False,
    },
    timeout=120,
)
data = response.json()
print(data["response"])

Python имеет встроенную поддержку многопоточности, что делает его легким для многозадачного программирования.


In [24]:

def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""
    response = httpx.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    return data["response"].strip()

In [26]:
answer = generate_answer("Что такое RAG в одном предложении?")
print(answer)

RAG stands for "Ranking, Aggregation, and Gating." It's a method used in natural language processing (NLP) for combining multiple sources of information into a single coherent summary. Here's a brief explanation of each component:

1. Ranking: This involves assigning a score to each piece of information based on its relevance to the overall topic or question.

2. Aggregation: This combines the information from different sources into a single, cohesive summary. It's similar to the idea of synthesizing or synthesizing information.

3. Gating: This is a technique used to select the most important or relevant information from the combined source. It's like filtering out the noise and focusing on the key points.

In simpler terms, RAG helps in creating a concise summary that captures the essence of multiple sources of information, ensuring that the final output is not just a list of facts but an informative and insightful summary.
